In [ ]:
!git clone https://github.com/zimarthur/ejection-fraction-us-model.git

In [ ]:
!pip install nibabel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys

sys.path.append("/content/ejection-fraction-us-model")
sys.path.append("/content/drive/MyDrive/Mestrado/Dissertacao/app/dataset")

In [ ]:
import torch
from torch import optim, nn
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm
import random
import os

from unet import UNet
from early_stopping import EarlyStopping
from data_loader import CamusSequenceDataset
from dice_bce_loss import DiceBCELoss

In [ ]:
# Copia a pasta inteira do Drive para o ambiente local do Colab
!cp -r /content/drive/MyDrive/Mestrado/Dissertacao/app/dataset /content/dataset_local



In [ ]:
LEARNING_RATE = 3e-4
BATCH_SIZE = 16
EPOCHS = 100
MAX_PATIENTS = None

MODEL_SAVE_PATH = "/content/drive/MyDrive/Mestrado/Dissertacao/app/results/unet.pth"
DATASET_PATH = "/content/dataset_local"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando dispositivo:", device)

In [ ]:
all_files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.pt')]
unique_patients = list(set([f.split('_')[0] for f in all_files]))
unique_patients.sort()

# 2. Embaralha a lista de pacientes (usamos uma seed para reprodutibilidade, se quiser)
random.seed(42)
random.shuffle(unique_patients)

if MAX_PATIENTS:
    unique_patients = unique_patients[:MAX_PATIENTS]
    print(f"⚠️ Usando apenas {len(unique_patients)} pacientes.")
else:
    print(f"🚀 Usando todos os {len(unique_patients)} pacientes.")

# 3. Divide a lista (ex: 80% treino, 20% validação)
train_ratio = 0.8
split_idx = int(len(unique_patients) * train_ratio)

train_patients = unique_patients[:split_idx]
val_patients = unique_patients[split_idx:]

print(f"Total de pacientes na pasta: {len(unique_patients)}")
print(f"Pacientes para Treino: {len(train_patients)} | Validação: {len(val_patients)}")

# 4. Instancia os datasets passando as listas autorizadas
train_dataset = CamusSequenceDataset(data_dir=DATASET_PATH, allowed_patients=train_patients)
val_dataset = CamusSequenceDataset(data_dir=DATASET_PATH, allowed_patients=val_patients)

In [ ]:
train_dataloader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dataloader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
model = UNet(in_channels=1, num_classes=1).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

criterion = DiceBCELoss().to(device)

early_stopping = EarlyStopping(patience=10, verbose=True, path=MODEL_SAVE_PATH)
scaler = torch.amp.GradScaler('cuda')

In [ ]:
for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_running_loss = 0
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for img, target in pbar:
        img, target = img.to(device, non_blocking=True), target.to(device, non_blocking=True)

        # Backward (Zero gradients no início)
        optimizer.zero_grad(set_to_none=True)

        # 2. O Autocast envolve apenas o Forward Pass e o cálculo da Loss
        with torch.amp.autocast('cuda'):
            y_pred = model(img)
            loss = criterion(y_pred, target) 

        scaler.scale(loss).backward()
        
        scaler.step(optimizer)
        
        scaler.update()

        train_running_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for img, target in val_dataloader:
            img, target = img.to(device, non_blocking=True), target.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                y_pred = model(img)
                loss = criterion(y_pred, target)
                
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_dataloader)
    avg_train_loss = train_running_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    scheduler.step(avg_val_loss)
    
    early_stopping(avg_val_loss, model)
    
    if early_stopping.early_stop:
        print("Early stopping acionado! Treinamento interrompido.")
        break